In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB


In [ ]:
!pip install -q vllm
!pip uninstall -y -q torchaudio

In [ ]:
import ctypes, glob, os, time
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
_cudart_paths = glob.glob("/usr/**/libcudart.so*", recursive=True)
_loaded = [ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL) for p in _cudart_paths]
print("preloaded:", _cudart_paths)

In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", gpu_memory_utilization=0.85, max_model_len=2048, enable_prefix_caching=True)
print("model loaded")

In [ ]:
TOOL_SCHEMA = {"type": "object", "properties": {"thought": {"type": "string"}, "tool": {"type": "string", "enum": ["calculator", "get_weather", "none"]}, "tool_input": {"type": "string"}, "final_answer": {"type": "string"}}, "required": ["thought", "tool", "tool_input", "final_answer"]}
def tool_calculator(expr): return str(eval(expr, {"__builtins__": {}}, {}))
def tool_get_weather(city): time.sleep(1.5); return f"{city}: 25C, sunny"
TOOLS = {"calculator": tool_calculator, "get_weather": tool_get_weather}
AGENT_SYS = "You are an agent. Tools: calculator(expr), get_weather(city). Respond ONLY as JSON with fields thought, tool (calculator|get_weather|none), tool_input, final_answer. Set tool=none and fill final_answer only when you have all info you need."

In [ ]:
prompts_5 = [AGENT_SYS + "\nUser: What's the weather in Seoul?\nAssistant:"] * 5
def looks_valid_json(t): return ('"tool"' in t) and ('"final_answer"' in t)
sp_free = SamplingParams(temperature=0.7, max_tokens=80)
outs_free = llm.generate(prompts_5, sp_free)
valid_free = sum(1 for t in [o.outputs[0].text for o in outs_free] if looks_valid_json(t))
print(f"No guided decoding: {valid_free}/5 look like valid tool-call JSON")

No guided decoding: 0/5 look like valid tool-call JSON


In [ ]:
from vllm.sampling_params import StructuredOutputsParams
so = StructuredOutputsParams(json=TOOL_SCHEMA)
sp_guided = SamplingParams(temperature=0.7, max_tokens=80, structured_outputs=so)
outs_guided = llm.generate(prompts_5, sp_guided)
valid_guided = sum(1 for t in [o.outputs[0].text for o in outs_guided] if looks_valid_json(t))
print(f"With guided decoding: {valid_guided}/5 look like valid tool-call JSON")

With guided decoding: 5/5 look like valid tool-call JSON


In [ ]:
import json
TASK = "Check the weather in Seoul, then if needed use the calculator to compute 15 times 23, then give me a final answer summarizing both."
history = ""; t_llm_total = 0.0; t_tool_total = 0.0; n_llm_calls = 0; n_tool_calls = 0
prompt_a = AGENT_SYS + "\nUser: " + TASK + "\nAssistant:"
t0 = time.time(); out_a = llm.generate([prompt_a], sp_guided); t_llm_total += time.time() - t0; n_llm_calls += 1
text_a = out_a[0].outputs[0].text; parsed_a = json.loads(text_a[text_a.find("{"):text_a.rfind("}")+1]); tool_a = parsed_a.get("tool", "none")
t0 = time.time(); result_a = TOOLS[tool_a](parsed_a.get("tool_input","")) if tool_a in TOOLS else None; t_tool_total += (time.time()-t0) if tool_a in TOOLS else 0.0; n_tool_calls += (1 if tool_a in TOOLS else 0)
print(f"Turn A: tool={tool_a} input={parsed_a.get('tool_input')} result={result_a}")
history += f"\nUser: {TASK}\nAssistant: {text_a}\nTool result: {result_a}"
prompt_b = AGENT_SYS + history + "\nUser: Continue.\nAssistant:"
t0 = time.time(); out_b = llm.generate([prompt_b], sp_guided); t_llm_total += time.time() - t0; n_llm_calls += 1
text_b = out_b[0].outputs[0].text; parsed_b = json.loads(text_b[text_b.find("{"):text_b.rfind("}")+1]); tool_b = parsed_b.get("tool", "none")
t0 = time.time(); result_b = TOOLS[tool_b](parsed_b.get("tool_input","")) if tool_b in TOOLS else None; t_tool_total += (time.time()-t0) if tool_b in TOOLS else 0.0; n_tool_calls += (1 if tool_b in TOOLS else 0)
print(f"Turn B: tool={tool_b} input={parsed_b.get('tool_input')} result={result_b}")
history += f"\nAssistant: {text_b}\nTool result: {result_b}"
prompt_c = AGENT_SYS + history + "\nUser: Continue.\nAssistant:"
t0 = time.time(); out_c = llm.generate([prompt_c], sp_guided); t_llm_total += time.time() - t0; n_llm_calls += 1
text_c = out_c[0].outputs[0].text; parsed_c = json.loads(text_c[text_c.find("{"):text_c.rfind("}")+1])
print(f"Turn C: tool={parsed_c.get('tool')} final_answer={parsed_c.get('final_answer')}")
total_wall = t_llm_total + t_tool_total
print(f"LLM inference (GPU-bound): {t_llm_total:.3f}s ({100*t_llm_total/total_wall:.1f}%) across {n_llm_calls} calls")
print(f"Tool execution (CPU/IO-bound): {t_tool_total:.3f}s ({100*t_tool_total/total_wall:.1f}%) across {n_tool_calls} calls")

Turn A: tool=calculator input=15 result=15
Turn B: tool=get_weather input=Seoul result=Seoul: 25C, sunny
Turn C: tool=get_weather final_answer=27
LLM inference (GPU-bound): 2.763s (64.8%) across 3 calls
Tool execution (CPU/IO-bound): 1.501s (35.2%) across 2 calls
